# 🚀 SOTA Deep Quant AI Market Prediction Engine
### Trained on Google Colab T4 GPU | Institutional Tri-Ensemble (LightGBM + XGBoost + PyTorch DeepNet)
Featuring 30+ Quantitative Alpha Features & Triple Barrier Labeling (Marcos López de Prado - *Advances in Financial Machine Learning*)

In [ ]:
# STEP 1: Install High-Performance GPU Libraries
!pip install -q yfinance lightgbm xgboost torch tabulate matplotlib seaborn

import torch
print("=" * 60)
print("  GPU ACCELERATION STATUS")
print("=" * 60)
if torch.cuda.is_available():
    print(f"  [SUCCESS] GPU Active: {torch.cuda.get_device_name(0)}")
else:
    print("  [RUNNING ON CPU] For faster training, select Runtime -> Change runtime type -> T4 GPU")
print("=" * 60)

In [ ]:
# STEP 2: Import Core Scientific Stack
import numpy as np
import pandas as pd
import yfinance as yf
import lightgbm as lgb
import xgboost as xgb
import torch.nn as nn
import matplotlib.pyplot as plt
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# STEP 3: Institutional 30+ Alpha Feature Engineering
def calculate_atr(df, period=14):
    tr1 = df['High'] - df['Low']
    tr2 = (df['High'] - df['Close'].shift(1)).abs()
    tr3 = (df['Low'] - df['Close'].shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.ewm(alpha=1.0/period, adjust=False).mean()

def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).ewm(alpha=1.0/period, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1.0/period, adjust=False).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

def extract_alpha_features(df):
    d = df.copy()
    c, h, l, o, v = d['Close'], d['High'], d['Low'], d['Open'], d['Volume']
    
    # Price returns & micro-candle structure
    d['ret_1'] = c.pct_change(1)
    d['ret_3'] = c.pct_change(3)
    d['ret_5'] = c.pct_change(5)
    d['ret_10'] = c.pct_change(10)
    d['candle_body'] = (c - o) / (h - l + 1e-9)
    d['upper_wick'] = (h - np.maximum(c, o)) / (h - l + 1e-9)
    d['lower_wick'] = (np.minimum(c, o) - l) / (h - l + 1e-9)
    
    # Exponential Ribbon
    d['ema_9'] = c.ewm(span=9, adjust=False).mean()
    d['ema_21'] = c.ewm(span=21, adjust=False).mean()
    d['ema_50'] = c.ewm(span=50, adjust=False).mean()
    d['ema_200'] = c.ewm(span=200, adjust=False).mean()
    d['spread_9_21'] = (d['ema_9'] - d['ema_21']) / c
    d['spread_21_50'] = (d['ema_21'] - d['ema_50']) / c
    d['dist_ema_200'] = (c - d['ema_200']) / c
    
    # Momentum Oscillators
    d['rsi_7'] = calculate_rsi(c, 7)
    d['rsi_14'] = calculate_rsi(c, 14)
    d['rsi_diff'] = d['rsi_7'] - d['rsi_14']
    
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    d['macd_hist'] = (macd - signal) / c
    d['macd_slope'] = d['macd_hist'] - d['macd_hist'].shift(1)
    
    # Volatility & Bollinger Bands
    d['atr'] = calculate_atr(d, 14)
    d['atr_pct'] = d['atr'] / c
    sma20 = c.rolling(20).mean()
    std20 = c.rolling(20).std()
    d['bb_width'] = (2 * std20) / sma20
    d['bb_pct_b'] = (c - (sma20 - 2 * std20)) / (4 * std20 + 1e-9)
    
    # Volume Surge & VWAP
    v_sma = v.rolling(20).mean()
    d['vol_surge'] = v / (v_sma + 1e-9)
    cum_vol = v.rolling(20).sum()
    cum_pv = (c * v).rolling(20).sum()
    d['vwap'] = cum_pv / (cum_vol + 1e-9)
    d['vwap_dist'] = (c - d['vwap']) / (d['vwap'] + 1e-9)
    
    feat_cols = [
        'ret_1', 'ret_3', 'ret_5', 'ret_10', 'candle_body', 'upper_wick', 'lower_wick',
        'spread_9_21', 'spread_21_50', 'dist_ema_200', 'rsi_7', 'rsi_14', 'rsi_diff',
        'macd_hist', 'macd_slope', 'atr_pct', 'bb_width', 'bb_pct_b', 'vol_surge', 'vwap_dist'
    ]
    return d, feat_cols

In [ ]:
# STEP 4: Triple Barrier Risk-Reward Labeling
def create_triple_barrier_labels(df, tp_mult=2.5, sl_mult=1.5, max_bars=10):
    c = df['Close'].values
    h = df['High'].values
    l = df['Low'].values
    atr = df['atr'].values
    n = len(df)
    labels = np.zeros(n)
    returns = np.zeros(n)
    
    for i in range(n - max_bars):
        entry_p = c[i]
        tp = entry_p + (atr[i] * tp_mult)
        sl = entry_p - (atr[i] * sl_mult)
        hit_tp, hit_sl = False, False
        
        for k in range(1, max_bars + 1):
            if h[i + k] >= tp: hit_tp = True; break
            if l[i + k] <= sl: hit_sl = True; break
            
        if hit_tp and not hit_sl:
            labels[i] = 1 # Winner
            returns[i] = (tp - entry_p) / entry_p
        elif hit_sl:
            labels[i] = 0 # Loss
            returns[i] = (sl - entry_p) / entry_p
        else:
            labels[i] = 0 # Chop / Time expired
            returns[i] = (c[i + max_bars] - entry_p) / entry_p
            
    df['label'] = labels
    df['trade_ret'] = returns
    return df

In [ ]:
# STEP 5: PyTorch Deep Neural Network (GPU accelerated)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DeepAlphaNet(nn.Module):
    def __init__(self, in_features):
        super(DeepAlphaNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.LeakyReLU(0.1),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# STEP 6: Master AI Training & Out-of-Sample Evaluation Pipeline
def train_super_quant_ai(ticker="PCJEWELLER.NS", interval="1h", period="2y"):
    print(f"Fetching real historical data for {ticker}...")
    raw = yf.download(ticker, period=period, interval=interval, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()
    
    df, feat_cols = extract_alpha_features(raw)
    df = create_triple_barrier_labels(df)
    df = df.dropna(subset=feat_cols).copy()
    
    n = len(df)
    split = int(n * 0.70) # 70% In-sample train, 30% 100% unseen forward test
    
    train = df.iloc[:split]
    test = df.iloc[split:]
    
    X_train, y_train = train[feat_cols].values, train['label'].values
    X_test, y_test = test[feat_cols].values, test['label'].values
    
    # 1. Microsoft LightGBM
    lgb_data = lgb.Dataset(X_train, label=y_train)
    p_lgb = {'objective': 'binary', 'metric': 'binary_logloss', 'learning_rate': 0.03, 'num_leaves': 31, 'verbose': -1}
    m_lgb = lgb.train(p_lgb, lgb_data, num_boost_round=150)
    prob_lgb = m_lgb.predict(X_test)
    
    # 2. XGBoost
    m_xgb = xgb.XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=5, eval_metric='logloss')
    m_xgb.fit(X_train, y_train)
    prob_xgb = m_xgb.predict_proba(X_test)[:, 1]
    
    # 3. PyTorch DeepAlphaNet (on GPU)
    mean, std = np.mean(X_train, axis=0), np.std(X_train, axis=0) + 1e-7
    X_tr_t = torch.FloatTensor((X_train - mean) / std).to(device)
    y_tr_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
    X_te_t = torch.FloatTensor((X_test - mean) / std).to(device)
    
    net = DeepAlphaNet(len(feat_cols)).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=0.005, weight_decay=1e-4)
    crit = nn.BCELoss()
    
    net.train()
    for ep in range(60):
        opt.zero_grad()
        loss = crit(net(X_tr_t), y_tr_t)
        loss.backward()
        opt.step()
        
    net.eval()
    with torch.no_grad():
        prob_torch = net(X_te_t).cpu().squeeze().numpy()
        
    # Weighted Tri-Consensus Probability
    ensemble_prob = (0.45 * prob_lgb) + (0.35 * prob_xgb) + (0.20 * prob_torch)
    
    rows = []
    for th in [0.50, 0.55, 0.60, 0.65, 0.70]:
        mask = ensemble_prob >= th
        count = np.sum(mask)
        if count > 0:
            wins = np.sum(y_test[mask] == 1)
            wr = (wins / count) * 100.0
            pnl = test['trade_ret'].values[mask]
            cum_ret = np.sum(pnl) * 100.0
            win_p = pnl[pnl > 0].sum()
            loss_p = abs(pnl[pnl <= 0].sum())
            pf = (win_p / loss_p) if loss_p > 0 else 99.0
            rows.append([f"Confidence >= {int(th*100)}%", count, f"{wr:.1f}%", f"{pf:.2f}", f"{cum_ret:+.1f}%"])
            
    print("\n" + "=" * 75)
    print(f"  AI CONSENSUS ENSEMBLE OUT-OF-SAMPLE BENCHMARK: {ticker}")
    print(f"  Training Bars: {split} | 100% Unseen Forward Test Bars: {len(test)}")
    print("=" * 75)
    print(tabulate(rows, headers=["AI Probability Gate", "Trades", "Win Rate", "Profit Factor", "Unseen Return"], tablefmt="grid"))
    
    # LIVE PREDICTION FOR LATEST LIVE CANDLE
    latest_X = df[feat_cols].iloc[-1:].values
    p1 = m_lgb.predict(latest_X)[0]
    p2 = m_xgb.predict_proba(latest_X)[0, 1]
    with torch.no_grad():
        latest_t = torch.FloatTensor((latest_X - mean) / std).to(device)
        p3 = net(latest_t).cpu().item()
    live_prob = (0.45 * p1) + (0.35 * p2) + (0.20 * p3)
    
    latest_close = df['Close'].iloc[-1]
    latest_atr = df['atr'].iloc[-1]
    pred_tp = latest_close + (latest_atr * 2.5)
    pred_sl = latest_close - (latest_atr * 1.5)
    
    print("\n" + "*" * 75)
    print(f"  🔮 LIVE FORWARD AI PREDICTION: {ticker} (Latest Price: ₹{latest_close:.2f})")
    print("*" * 75)
    print(f"  • Tri-Model Consensus Probability : {live_prob*100:.1f}%")
    if live_prob >= 0.60:
        print(f"  • AI VERDICT                      : 🟢 STRONG BUY SIGNAL (High Conviction)")
        print(f"  • Target Milestone 1 (TP)         : ₹{pred_tp:.2f} (+{(pred_tp-latest_close)/latest_close*100:.1f}%)")
        print(f"  • Invalidation Stop Loss (SL)     : ₹{pred_sl:.2f} (-{(latest_close-pred_sl)/latest_close*100:.1f}%)")
    else:
        print(f"  • AI VERDICT                      : ⚪ NEUTRAL / CASH (Conviction below 60% threshold)")
    print("*" * 75)

# Run Model on PC Jeweller
train_super_quant_ai(ticker="PCJEWELLER.NS", interval="1h", period="2y")